# 06 · Scoring the copy-paste twins — the controlled generator comparison

Closes **checklist item 3**: is the low synthetic sensitivity a detection failure or a
generation failure?

The copy-paste arm is the controlled experiment. Same source chests, same insertion sites,
same frozen detector, same 512 px canvas. **Only the rendering differs** — a real nodule
patch pasted in, versus a diffusion edit. So any sensitivity gap is attributable to how the
lesion was rendered, and no radiologist is required to make the argument.

`copypaste.csv` has 648 rows matched to the grid by `radedit_twin`, and `det_edited` is
**entirely NaN** — they were generated and never scored. That single gap is all that stands
between the project and a paired test.

## Preprocessing here is deliberately the grid's, not the detector's

Notebook 03 §9 establishes that the detector was trained on native-resolution
max-normalised images, and that 512+CLAHE is out of distribution. **This notebook uses
512+CLAHE anyway**, because the comparison being made is copy-paste *against the RadEdit
grid*, and the grid only exists at 512+CLAHE — RadEdit cannot run at any other resolution.

Holding preprocessing constant across the two arms is what makes the contrast valid. Both
arms are out of distribution by the same amount, so it cancels. What must not be done is
compare either arm's absolute sensitivity to the native real-data numbers from 03 §9.

CPU-light, one GPU pass over 648 images — about ten minutes.

## 1 · Paths and inputs

In [ ]:
# ===========================================================================
# CANONICAL DRIVE PATHS
# Mirrored from src/paths.py and notebooks/CONFIG_CELL.md. Mapped from Drive
# 2026-09-12 with the Drive connector. Change all three together.
# Full layout, folder ids and the old->new table: DRIVE_LAYOUT.md
# ===========================================================================
!pip -q install SimpleITK
import os, json, glob, time, shutil, subprocess, random
from pathlib import Path
from google.colab import drive

# --- mount -----------------------------------------------------------------
# ismount(), not isdir(). A plain local directory under an unmounted
# /content/drive is also a dir, and creating one blocks the mount and makes
# Drive look empty -- that happened once and looked like a wiped Drive.
if not os.path.ismount('/content/drive'):
    if Path('/content/drive').exists():
        os.system('fusermount -u /content/drive 2>/dev/null')
        shutil.rmtree('/content/drive', ignore_errors=True)
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'mount failed'

# --- resolve the root ------------------------------------------------------
# MyDrive/Algoverse is a SHORTCUT to the shared folder Feliciano_Algoverse.
# One folder, not two; the FUSE mount resolves it as a directory.
# Test for the 01_data marker, never for the root itself: an unresolved
# shortcut and a stale empty directory both "exist", and that is exactly how
# a stray empty results/ tree got created on 2026-09-12.
ROOT = None
for _c in ['/content/drive/MyDrive/Algoverse',
           '/content/drive/MyDrive/Feliciano_Algoverse',
           '/content/drive/Shareddrives/Feliciano_Algoverse']:
    if (Path(_c)/'01_data').is_dir():
        ROOT = Path(_c); break
assert ROOT is not None, (
    'Algoverse root not found. Tried MyDrive/Algoverse, '
    'MyDrive/Feliciano_Algoverse, Shareddrives/Feliciano_Algoverse.\n'
    f'MyDrive top level: '
    f'{sorted(p.name for p in Path("/content/drive/MyDrive").iterdir())[:20]}\n'
    'MyDrive/Algoverse is a shortcut to the shared Feliciano_Algoverse. If it '
    'is gone: Drive -> Shared with me -> right-click Feliciano_Algoverse -> '
    'Add shortcut to Drive -> My Drive.')

# --- layout ----------------------------------------------------------------
SOURCE    = ROOT/'01_data'/'00_source'           # node21, chexpert, mimic_cxr
NODE21    = SOURCE/'node21'
MHA_SRC   = NODE21/'images'                      # 4,882 .mha
ANN_CSV   = NODE21/'metadata.csv'                # 5,224 rows, 1,476 label==1
GRID      = ROOT/'01_data'/'01_grid'
GRID_CSV  = GRID/'grid_v5.csv'                   # 231,145 bytes if it is the right one
RUNS      = GRID/'_runs'                         # generation checkpoint zips, not data
EMB_DIR   = ROOT/'01_data'/'02_embeddings'
CPASTE    = ROOT/'01_data'/'03_copypaste'
MODELS    = ROOT/'02_results'/'00_models'
CKPT      = MODELS/'baseline1_checkpoint.pth'    # .pth -- the old .pt path is dead
FIGS      = ROOT/'02_results'/'01_figures'
B1_DIR    = ROOT/'02_results'/'02_baselines'
PRED_DIR  = ROOT/'02_results'/'03_predictor'
B3_DIR    = ROOT/'02_results'/'04_baseline3'

print(f'root: {ROOT}')

OUT  = Path('/content/copypaste'); OUT.mkdir(parents=True, exist_ok=True)
DEST = PRED_DIR.parent/'05_copypaste'; (DEST).mkdir(parents=True, exist_ok=True)
for _p in (CKPT, CPASTE/'copypaste.csv', CPASTE/'images', GRID_CSV):
    assert _p.exists(), f'{_p} missing -- see DRIVE_LAYOUT.md'
print(f'inputs ok\nwriting to {DEST}')

In [ ]:
import numpy as np, pandas as pd, cv2, torch, torchvision
from PIL import Image
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF

cp = pd.read_csv(CPASTE/'copypaste.csv', keep_default_na=False, na_values=[''])
g  = pd.read_csv(GRID_CSV, keep_default_na=False, na_values=[''])
print(f'copypaste: {len(cp)} rows, {cp.shape[1]} cols')
print(f'columns: {list(cp.columns)}')
print(f'\ndet_edited non-null: {cp.det_edited.notna().sum()} of {len(cp)}'
      if 'det_edited' in cp.columns else '\nno det_edited column yet')
assert 'radedit_twin' in cp.columns, 'no radedit_twin column -- cannot pair'
print(f'twins resolvable against the grid: '
      f'{cp.radedit_twin.isin(set(g.image_id)).sum()} of {len(cp)}')
imgs = sorted((CPASTE/'images').glob('*.png'))
print(f'{len(imgs)} png on disk')

## 2 · Detector — identical settings to the grid run

`box_score_thresh=0.0` and a 300-box cap, so every box is stored and no later threshold or
matching-rule sweep needs inference again. `SCORE_MIN` is the operating point at which a
lesion counts as detected, **not** a filter.

In [ ]:
SIZE, DET_SIZE, SCORE_MIN = 512, 800, 0.05
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

DET = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights=None, weights_backbone=None,
    box_score_thresh=0.0, box_detections_per_img=300)
DET.roi_heads.box_predictor = FastRCNNPredictor(
    DET.roi_heads.box_predictor.cls_score.in_features, 2)
_st = torch.load(CKPT, map_location=DEV, weights_only=False)
DET.load_state_dict(_st['model'] if isinstance(_st, dict) and 'model' in _st else _st)
DET = DET.eval().to(DEV)
print(f'detector on {DEV}, from {CKPT.name}')


@torch.no_grad()
def detect_all(arr01, size=DET_SIZE):
    im = Image.fromarray((arr01*255).astype(np.uint8)).convert('RGB') \
              .resize((size, size), Image.LANCZOS)
    o = DET([TF.to_tensor(im).to(DEV)])[0]
    return o['boxes'].cpu().numpy()/size, o['scores'].cpu().numpy()


def centre_in(b, t):
    cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
    return t[0] <= cx <= t[2] and t[1] <= cy <= t[3]


def best_at(b, sc, t):
    return float(max((ss for bb, ss in zip(b, sc) if centre_in(bb, t)), default=0.0))

## 3 · Score, resumable

One JSON per image synced to Drive every 50, so a dead session costs 50 images rather than
the run. Box columns are read from `copypaste.csv`; the cell asserts on their presence
rather than guessing a name.

In [ ]:
import json, time, shutil
(OUT/'dets').mkdir(parents=True, exist_ok=True)
(DEST/'dets').mkdir(parents=True, exist_ok=True)
for f in (DEST/'dets').iterdir():
    if not (OUT/'dets'/f.name).exists():
        shutil.copy(f, OUT/'dets'/f.name)


def sync():
    n = 0
    for f in (OUT/'dets').iterdir():
        if not (DEST/'dets'/f.name).exists():
            shutil.copy(f, DEST/'dets'/f.name); n += 1
    return n


IDCOL = 'image_id' if 'image_id' in cp.columns else cp.columns[0]
BOXCOLS = None
for cand in [('mask_x0','mask_y0','mask_x1','mask_y1'), ('x0','y0','x1','y1'),
             ('box_x0','box_y0','box_x1','box_y1')]:
    if all(q in cp.columns for q in cand):
        BOXCOLS = cand; break
assert BOXCOLS, f'no recognisable box columns in {list(cp.columns)}'
print(f'id column: {IDCOL}   box columns: {BOXCOLS}')

done = {p.stem for p in (OUT/'dets').glob('*.json')}
todo = [r for r in cp.itertuples() if getattr(r, IDCOL) not in done]
print(f'{len(done)} scored, {len(todo)} to go')

t0 = time.time()
for i, r in enumerate(todo):
    iid = getattr(r, IDCOL)
    p = CPASTE/'images'/f'{iid}.png'
    assert p.exists(), f'{p} missing -- refusing to skip silently'
    a = np.asarray(Image.open(p).convert('L'), np.float32)/255.0
    if a.shape != (SIZE, SIZE):
        a = cv2.resize(a, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    b, sc = detect_all(a)
    json.dump({'boxes': b.tolist(), 'scores': sc.tolist()},
              open(OUT/'dets'/f'{iid}.json', 'w'))
    if (i+1) % 50 == 0 or i == len(todo)-1:
        el = time.time()-t0
        print(f'  {i+1}/{len(todo)}  {el/60:5.1f} min  '
              f'~{el/(i+1)*(len(todo)-i-1)/60:4.0f} min left  +{sync()} to Drive')
print(f'\n{len(list((OUT/"dets").glob("*.json")))} files, {sync()} synced')

## 4 · Fill in `det_edited` and pair against the grid

In [ ]:
rows = []
for r in cp.itertuples():
    iid = getattr(r, IDCOL)
    f = OUT/'dets'/f'{iid}.json'
    if not f.exists():
        continue
    d = json.load(open(f))
    b, sc = np.array(d['boxes']).reshape(-1, 4), np.array(d['scores'])
    t = tuple(getattr(r, q) for q in BOXCOLS)
    rows.append(dict(image_id=iid, radedit_twin=r.radedit_twin,
                     det_cp=round(best_at(b, sc, t), 4),
                     n_boxes_cp=int(len(sc))))
CP = pd.DataFrame(rows)
assert len(CP), 'nothing scored'
print(f'{len(CP)} copy-paste images scored; det_cp>0 on {(CP.det_cp > 0).sum()}')

gi = g.set_index('image_id')
CP['det_radedit'] = CP.radedit_twin.map(gi.det_edited)
CP['cnr_radedit'] = CP.radedit_twin.map(gi.cnr)
CP['chest']       = CP.radedit_twin.map(gi.chest)
PAIR = CP.dropna(subset=['det_radedit']).copy()
print(f'{len(PAIR)} pairs resolvable, {PAIR.chest.nunique()} chests')

PAIR['hit_cp'] = PAIR.det_cp      >= SCORE_MIN
PAIR['hit_re'] = PAIR.det_radedit >= SCORE_MIN
CP.to_csv(OUT/'copypaste_scored.csv', index=False)
shutil.copy(OUT/'copypaste_scored.csv', DEST)
print(f'\ncopy-paste sensitivity @{SCORE_MIN}: {PAIR.hit_cp.mean():.4f} '
      f'({PAIR.hit_cp.sum()}/{len(PAIR)})')
print(f'RadEdit    sensitivity @{SCORE_MIN}: {PAIR.hit_re.mean():.4f} '
      f'({PAIR.hit_re.sum()}/{len(PAIR)})')
print(f'difference: {PAIR.hit_cp.mean()-PAIR.hit_re.mean():+.4f}')

## 5 · The paired test, with chest as the unit of independence

Two tests, because they answer different questions and the second is the one to report.

**McNemar** on the discordant pairs treats each twin pair as independent. With multiple
images per chest that is pseudoreplication — the same objection raised everywhere else in
this project.

**Cluster bootstrap over chests** resamples the 12 source chests with replacement. Twelve
clusters is few, so the interval will be wide; that is the honest width, not a defect.

In [ ]:
from scipy.stats import binomtest

b01 = int((~PAIR.hit_cp &  PAIR.hit_re).sum())
b10 = int(( PAIR.hit_cp & ~PAIR.hit_re).sum())
print(f'discordant pairs: copy-paste only {b10}, RadEdit only {b01}')
if b01+b10:
    mc = binomtest(b10, b01+b10, 0.5)
    print(f'McNemar exact (pairs as independent): p={mc.pvalue:.6f}')
else:
    print('no discordant pairs')

rng = np.random.default_rng(0)
chests = PAIR.chest.unique()
per = {c: PAIR[PAIR.chest == c] for c in chests}
boot = []
for _ in range(4000):
    pick = rng.choice(chests, size=len(chests), replace=True)
    s = pd.concat([per[c] for c in pick])
    boot.append(s.hit_cp.mean() - s.hit_re.mean())
lo, hi = np.percentile(boot, [2.5, 97.5])
obs = PAIR.hit_cp.mean() - PAIR.hit_re.mean()
print(f'\nchest-clustered bootstrap over {len(chests)} chests:')
print(f'  difference {obs:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]')
print(f'  excludes zero: {bool(lo > 0 or hi < 0)}')

per_chest = PAIR.groupby('chest').agg(n=('hit_cp', 'size'), cp=('hit_cp', 'mean'),
                                      re=('hit_re', 'mean'))
per_chest['delta'] = (per_chest.cp - per_chest.re).round(4)
print('\nper chest:'); print(per_chest.round(3).to_string())

OUTT = pd.DataFrame([dict(n_pairs=len(PAIR), n_chests=len(chests),
                          sens_copypaste=round(PAIR.hit_cp.mean(), 4),
                          sens_radedit=round(PAIR.hit_re.mean(), 4),
                          difference=round(obs, 4), ci_lo=round(lo, 4), ci_hi=round(hi, 4),
                          mcnemar_p=(binomtest(b10, b01+b10, 0.5).pvalue
                                     if b01+b10 else np.nan),
                          discordant_cp_only=b10, discordant_re_only=b01,
                          operating_point=SCORE_MIN)])
OUTT.to_csv(OUT/'table-generator-comparison.csv', index=False)
per_chest.to_csv(OUT/'table-generator-comparison-per-chest.csv')
for f in ['table-generator-comparison.csv', 'table-generator-comparison-per-chest.csv']:
    shutil.copy(OUT/f, DEST)
print()
print(OUTT.to_string(index=False))

print('\nWHAT IT MEANS:')
print('  copy-paste clearly higher -> the gap is GENERATOR RENDERING, since chests, sites,')
print('     detector and preprocessing are all held constant. The synthetic sensitivity')
print('     characterises the generator-detector pair, not the detector. Checklist item 3')
print('     closes, and the copy-paste rate is the ceiling insertion realism allows.')
print('  the two are close        -> rendering is not the bottleneck and the low synthetic')
print('     sensitivity is about the lesions themselves being unconspicuous at 512 px,')
print('     which is the same story as the CNR result in FINDINGS section 1.')
print(f'\nsynced {sync()} detections. Tables at {DEST}')